# 01 — Classical GAN Baseline (SMI log-returns)

**ZHAW CEC Quantum Computing — Final project**

Trains a small classical GAN on Swiss Market Index log-return windows.

**Switch between vanilla GAN and WGAN-GP via `MODEL_VARIANT`.**

## Setup (Colab)

In [ ]:
# Run on Colab; safe to re-run.
import os
if not os.path.isdir('/content/qGAN-market-generator'):
    !git clone https://github.com/wuns/qGAN-market-generator.git
%cd /content/qGAN-market-generator
!pip install -q yfinance pennylane

In [ ]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd()
if (ROOT / 'src').is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'src').is_dir():
    sys.path.insert(0, str(ROOT.parent))
    ROOT = ROOT.parent

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data       import prepare_smi_data
from src.models     import ClassicalGenerator, Discriminator, Critic, count_parameters
from src.training   import set_seed, make_dataloader, build_experiment, generate
from src.evaluation import (plot_distributions, plot_acf_comparison,
                            plot_sample_paths, build_report)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device, '| repo root =', ROOT)

## Config

**The switch.** Set `MODEL_VARIANT` to either `'gan'` or `'wgan_gp'`. Everything below adapts automatically.

In [ ]:
MODEL_VARIANT = 'wgan_gp'   # 'gan'  or  'wgan_gp'

SEED       = 42
WINDOW     = 20
LATENT_DIM = 8
EPOCHS     = 80
BATCH      = 64

set_seed(SEED)

## Data

In [ ]:
data = prepare_smi_data(
    window=WINDOW,
    cache_path=ROOT / 'results' / 'smi_prices.pkl',
)
print(f'Train windows: {data.train_windows.shape}')
print(f'Test  windows: {data.test_windows.shape}')
print(f'Scale (4 sigma): {data.scale:.5f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(data.raw_returns, lw=0.4); ax[0].set_title('SMI log-returns'); ax[0].set_ylabel('r_t')
ax[1].hist(data.raw_returns, bins=80); ax[1].set_title('Distribution'); ax[1].set_xlabel('r_t')
plt.tight_layout(); plt.show()

## Model & training

`build_experiment` returns the right generator, adversary (Discriminator or Critic), and training function for the chosen variant. The notebook stays the same.

In [ ]:
# Pick the adversary class to match the variant. Both have identical architecture
# in this project, so the comparison is genuinely controlled.
adversary_cls = Discriminator if MODEL_VARIANT == 'gan' else Critic

exp = build_experiment(
    variant=MODEL_VARIANT,
    latent_dim=LATENT_DIM,
    window=WINDOW,
    generator_cls=ClassicalGenerator,
    adversary_cls=adversary_cls,
)

n_params_G = count_parameters(exp.generator)
n_params_A = count_parameters(exp.adversary)
print(f'Variant:           {exp.label}')
print(f'Generator params:  {n_params_G}')
print(f'{exp.adversary_role.capitalize():18s} params: {n_params_A}')

loader = make_dataloader(data.train_windows, batch_size=BATCH)
history = exp.train_fn(
    exp.generator, exp.adversary, loader,
    latent_dim=LATENT_DIM, epochs=EPOCHS, device=device,
)
print(f'\nTraining time: {history.train_time_sec:.1f} s')

In [ ]:
plt.figure(figsize=(8, 3))
d_label = 'D loss' if exp.adversary_role == 'discriminator' else 'C loss'
plt.plot(history.d_loss, label=d_label)
plt.plot(history.g_loss, label='G loss')
plt.xlabel('epoch'); plt.legend(); plt.title(f'{exp.label} training losses')
plt.tight_layout(); plt.show()

## Generate & evaluate

In [ ]:
n_eval = len(data.test_windows)
fake_scaled, t_inf = generate(exp.generator, n_eval, LATENT_DIM, device=device)
fake_returns = data.unscale(fake_scaled)
real_returns = data.unscale(data.test_windows)
samples_per_sec = (n_eval * WINDOW) / t_inf
print(f'Inference: {n_eval} windows in {t_inf*1000:.1f} ms ({samples_per_sec:.0f} samples/s)')

In [ ]:
plot_sample_paths(real_returns, fake_returns); plt.tight_layout(); plt.show()
plot_distributions(real_returns, fake_returns); plt.tight_layout(); plt.show()
plot_acf_comparison(real_returns, fake_returns); plt.tight_layout(); plt.show()

In [ ]:
report = build_report(
    real_returns=real_returns,
    fake_returns=fake_returns,
    n_params_G=n_params_G,
    n_params_D=n_params_A,
    train_time_sec=history.train_time_sec,
    inference_samples_per_sec=samples_per_sec,
    extras={'seed': SEED, 'epochs': EPOCHS, 'window': WINDOW,
            'latent_dim': LATENT_DIM, 'model': f'classical_{MODEL_VARIANT}'},
)
for k, v in report.items():
    print(f'{k:30s} {v}')

## Save artefacts

Each variant saves to its own subfolder so you can keep results from both for comparison.

In [ ]:
out = ROOT / 'results' / f'classical_{MODEL_VARIANT}'
out.mkdir(parents=True, exist_ok=True)
torch.save(exp.generator.state_dict(), out / 'generator.pt')
np.save(out / 'fake_returns.npy', fake_returns)
np.save(out / 'real_returns_test.npy', real_returns)
np.save(out / 'scale.npy', np.array([data.scale]))
with open(out / 'metrics.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)
print('Saved to', out)

## How to run both variants

1. Set `MODEL_VARIANT = 'gan'`, **Runtime → Run all**, look at metrics & plots.
2. Set `MODEL_VARIANT = 'wgan_gp'`, run all again.
3. Both result folders (`results/classical_gan/` and `results/classical_wgan_gp/`) now hold their respective `metrics.json` and samples — `03_comparison.ipynb` will load them side by side.